# Model Comparison and Evaluation

This notebook compares the performance of different machine learning models for predicting student dropout using cross-validation methodology. All models are trained and evaluated on the same standardized dataset to ensure fair comparison.

## Models:
- **Logistic Regression** - Linear baseline with feature scaling
- **Decision Tree** - Interpretable non-linear model
- **Random Forest** - Ensemble method for robust predictions
- **XGBoost** - Gradient boosting ensemble method
- **Neural Network (MLP)** - Non-linear deep learning approach

## Metrics Evaluated:
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC

## Table of Contents

1. **[Import Libraries & Load Data](#1-import-libraries--load-data)** - Load necessary Python packages and dataset
2. **[Data Preparation](#2-data-preparation)** - Split data and prepare features and target variables for cross-validation
3. **[Model Configuration](#3-model-configuration)** - Configure all four models with balanced class weights
4. **[Cross-Validation Evaluation](#4-cross-validation-evaluation)** - Train and evaluate all models using 10-fold CV
5. **[Model Comparison Dashboard](#5-model-comparison-dashboard)** - Compare performance metrics across models
6. **[Generalization of models](#6-generalization-of-models)** - Evaluate model generalization and performance on unseen data
7. **[Feature Importance Comparison](#7-feature-importance-comparison)** - Compare feature importance across models
8. **[Best Model Selection & Comprehensive Analysis](#8-best-model-selection--comprehensive-analysis)** - Select best model, analyze performance, and provide recommendations
9. **[Save data and CV results](#9-save-data-and-cv-results)** - Save trained models and cross-validation results for future use

## 1. Import Libraries & Load Data

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')
import joblib

# Scikit-learn imports
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_validate
from sklearn.base import clone  
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score, 
    precision_score, f1_score,
    average_precision_score  # // Added for PR-AUC (Average Precision)
)
from sklearn.utils.class_weight import compute_sample_weight

# Wilcoxon signed-rank test imports
from scipy.stats import wilcoxon
from itertools import combinations

# XGBoost import
from xgboost import XGBClassifier

# Imbalanced-learn imports
from imblearn.over_sampling import SMOTE

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# Load the dataset
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
df = pd.read_csv(PROCESSED_DIR / 'feature_selection_all.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nDataset info:")
print(df.info())

In [ ]:
# Define parameter values
OVERFITTING_THRESHOLD = 0.05
STABILITY_THRESHOLD = 0.02
N_FOLDS = 10
N_REPEATS = 5

## 2. Data Preparation

**Purpose**: Split data into train/validation sets and prepare features and target variables for cross-validation.

In [ ]:
# Specify the target variable
target_col = 'sdo5_degree_drop_out'

# Split data based on the 'set' column
train_df = df[df['set'] == 'train'].copy()
val_df = df[df['set'] == 'val'].copy()
test_df = df[df['set'] == 'test'].copy()

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

In [ ]:
# Define target and feature columns
exclude_cols = ['set', target_col]
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Target column: {target_col}")
print(f"Number of features: {len(feature_cols)}")
print(f"Excluded columns: {exclude_cols}")

# Prepare feature matrices and target vectors
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

# Combine train and validation for cross-validation
X_train_val = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_train_val = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

print(f"\nCombined train+val shape for cross-validation: {X_train_val.shape}")

print("\nClass distribution in combined train+val set:")
print(y_train_val.value_counts(normalize=True))

In [ ]:
# Verify data quality (there should be no missing values)
missing_counts = X_train_val.isnull().sum().sum()
print(f"Missing values in features: {missing_counts}")
if missing_counts > 0:
    print("⚠️ Warning: Missing values found! Please perform imputation.")
    missing_cols = X_train_val.isnull().sum()[X_train_val.isnull().sum() > 0]
    print("Columns with missing values:")
    print(missing_cols)
else:
    print("✅ Data quality verified: No missing values found.")

## 3. Model Configuration

**Purpose**: Configure all five models with appropriate parameters and balanced class weights.

In [ ]:
# =============================================================================
# FEEDBACK ON MODEL CONFIGURATION AND CV SETUP
# -----------------------------------------------------------------------------
# A few important consistency and hygiene checks
#
# 1) Class imbalance handling
#    - Logistic Regression, Decision Tree, and Random Forest correctly rely on
#      class_weight='balanced'. This is handled per fold by scikit-learn and
#      does not require manual intervention.
#    - XGBoost currently uses a single, globally computed scale_pos_weight.
#      Given that cross-validation is performed on the combined train+val set,
#      this value should ideally be computed from the labels of the current
#      training fold rather than fixed once. A global value is an acceptable
#      approximation, but fold-specific computation would be cleaner and more
#      consistent with the rest of the setup.
#    - The Neural Network handles imbalance via SMOTE applied to the training
#      fold only. This is appropriate.
#
# 2) Consistency across models
#    - Different imbalance strategies are used across models (class weights,
#      scale_pos_weight, SMOTE). This is fine, but each strategy must remain
#      strictly fold-local and should not leak information from validation
#      data into training.
#
# 3) Random Forest configuration
#    - max_samples is used, which implicitly relies on bootstrap sampling.
#      Although bootstrap=True is the default, it would be safer to set this
#      explicitly.
#    - n_estimators is not explicitly set; fixing this improves stability and
#      reproducibility across runs.
#
# 4) Decision Tree and Random Forest regularization
#    - The current depth and minimum-sample constraints are quite restrictive.
#      These constraints are fine if intentional.
#
# 5) Logistic Regression configuration
#    - The solver is not explicitly specified. Defining it (e.g., liblinear or
#      saga) would improve reproducibility and convergence transparency.


In [ ]:
# Calculate baseline accuracy

# baseline is computed on train+val because CV is run on train+val.
baseline_accuracy = y_train_val.value_counts(normalize=True).max()
print(f"Baseline accuracy (majority class): {baseline_accuracy:.3f}")

In [ ]:
# Configure all models with balanced class weights
models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        class_weight='balanced',
        max_iter=1000,
        solver='liblinear'  
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        class_weight='balanced',
        max_depth=12,               # Moderate depth limit
        min_samples_split=30,       # Moderate split threshold
        min_samples_leaf=15,        # Moderate leaf threshold
        max_features='sqrt',        # Standard feature limitation
        min_impurity_decrease=0.005 # Small improvement requirement
        # Above parameters are quite restrictive, due to overfitting concerns
    ),
    'Random Forest': RandomForestClassifier(
        random_state=42,
        class_weight='balanced',
        n_jobs=-1,
        n_estimators=100,       # default
        bootstrap=True,         # default because max_samples is used
        max_depth=12,           # Moderate depth limit
        min_samples_split=30,   # Moderate split threshold
        min_samples_leaf=15,    # Moderate leaf threshold
        max_features='sqrt',    # Default feature limit
        max_samples=0.9         # Use 90% of samples
    ),
    'XGBoost': XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        verbosity=0,
        n_jobs=-1,
        max_depth=5,                        # Moderate depth limit
        learning_rate=0.08,                 # Slightly slower learning
        n_estimators=80,                    # Moderate number of trees
        reg_alpha=0.3,                      # Moderate L1 regularization
        reg_lambda=1.5,                     # Moderate L2 regularization
        subsample=0.85,                     # Use 85% of samples
        colsample_bytree=0.85,              # Use 85% of features
        min_child_weight=3                  # Moderate leaf requirement
        ),
    'Neural Network': MLPClassifier(
        random_state=42,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.1
    )
}

print("Model configurations:")
for name, model in models.items():    
    print(f"- {name}: {type(model).__name__}")


## 4. Cross-Validation
**Purpose**: Train and evaluate all models using stratified 10-fold cross-validation.

In [ ]:
# Set up cross-validation strategy
cv_strategy = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=42)

# Define scoring metrics
scoring_metrics = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',  # // Added PR-AUC (Average Precision) for dropout class
    'f1': 'f1',
    'recall': 'recall',
    'precision': 'precision'
}

print(f"Cross-validation strategy: {N_FOLDS}-fold stratified")
print(f"Scoring metrics: {list(scoring_metrics.keys())}")


In [ ]:
# Initialize results storage
cv_results = {}

print("Starting cross-validation for all models...\n")

for model_name, model in models.items():
    start_time = time.time()
    
    # Extract data
    X_for_cv = X_train_val.values
    
    if model_name == 'Neural Network':
        # Manual CV loop to apply SMOTE within each fold

        # Initialize results with consistent naming (matching cross_validate format)
        cv_scores = {
            "test_f1": [],
            "test_recall": [],
            "test_roc_auc": [],
            "test_pr_auc": [],        # // Added PR-AUC (Average Precision) on test fold
            "test_precision": [],
            "test_accuracy": [],
            "train_f1": [],
            "train_recall": [],
            "train_roc_auc": [],
            "train_pr_auc": [],       # // Added PR-AUC (Average Precision) on train fold
            "train_precision": [],
            "train_accuracy": []
        }

        for train_idx, test_idx in cv_strategy.split(X_train_val, y_train_val):
            
            # Split folds
            X_tr, y_tr = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
            X_te, y_te = X_train_val.iloc[test_idx], y_train_val.iloc[test_idx]
            
            # Apply SMOTE on training data only
            sm = SMOTE(random_state=42)
            X_res, y_res = sm.fit_resample(X_tr, y_tr)

            # Train the MLP
            model.fit(X_res, y_res)

            # Predict on test fold
            y_te_prob = model.predict_proba(X_te)[:, 1]
            y_te_pred = model.predict(X_te)

            # Predict on ORIGINAL training fold (for comparable train scores)
            y_tr_prob = model.predict_proba(X_tr)[:, 1]
            y_tr_pred = model.predict(X_tr)

            # Test metrics
            cv_scores["test_f1"].append(f1_score(y_te, y_te_pred, zero_division=0))
            cv_scores["test_recall"].append(recall_score(y_te, y_te_pred, zero_division=0))
            cv_scores["test_roc_auc"].append(roc_auc_score(y_te, y_te_prob))
            cv_scores["test_pr_auc"].append(average_precision_score(y_te, y_te_prob))  # // Added PR-AUC
            cv_scores["test_precision"].append(precision_score(y_te, y_te_pred, zero_division=0))
            cv_scores["test_accuracy"].append(accuracy_score(y_te, y_te_pred))

            # Train metrics (computed on original fold)
            cv_scores["train_accuracy"].append(accuracy_score(y_tr, y_tr_pred))
            cv_scores["train_pr_auc"].append(average_precision_score(y_tr, y_tr_prob))  # // Added PR-AUC

        # Convert lists to numpy arrays (matching cross_validate format)
        cv_scores = {k: np.array(v) for k, v in cv_scores.items()}

    elif model_name == 'XGBoost':
        # Manual CV loop so scale_pos_weight can be computed on the training fold only (fold-local imbalance handling).
        cv_scores = {
            "test_accuracy": [], "test_roc_auc": [], "test_pr_auc": [], "test_f1": [], "test_recall": [], "test_precision": [],  # // Added test_pr_auc
            "train_accuracy": [], "train_roc_auc": [], "train_pr_auc": [], "train_f1": [], "train_recall": [], "train_precision": []  # // Added train_pr_auc
        }

        for train_idx, test_idx in cv_strategy.split(X_train_val, y_train_val):
            X_tr, y_tr = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
            X_te, y_te = X_train_val.iloc[test_idx], y_train_val.iloc[test_idx]

            # Fold-specific scale_pos_weight computed from y_tr (not from y_train_val).
            neg = (y_tr == 0).sum()
            pos = (y_tr == 1).sum()
            spw = neg / pos if pos > 0 else 1.0  # // Added zero-division guard

            # Clone model so each fold starts clean; inject fold-specific spw.
            xgb_fold = clone(model).set_params(scale_pos_weight=spw)
            xgb_fold.fit(X_tr, y_tr)

            y_te_prob = xgb_fold.predict_proba(X_te)[:, 1]
            y_te_pred = xgb_fold.predict(X_te)

            y_tr_prob = xgb_fold.predict_proba(X_tr)[:, 1]
            y_tr_pred = xgb_fold.predict(X_tr)

            # Test metrics
            cv_scores["test_accuracy"].append(accuracy_score(y_te, y_te_pred))
            cv_scores["test_roc_auc"].append(roc_auc_score(y_te, y_te_prob))
            cv_scores["test_pr_auc"].append(average_precision_score(y_te, y_te_prob))  # // Added PR-AUC
            cv_scores["test_f1"].append(f1_score(y_te, y_te_pred, zero_division=0))
            cv_scores["test_recall"].append(recall_score(y_te, y_te_pred, zero_division=0))
            cv_scores["test_precision"].append(precision_score(y_te, y_te_pred, zero_division=0))

            # Train metrics
            cv_scores["train_accuracy"].append(accuracy_score(y_tr, y_tr_pred))
            cv_scores["train_roc_auc"].append(roc_auc_score(y_tr, y_tr_prob))
            cv_scores["train_pr_auc"].append(average_precision_score(y_tr, y_tr_prob))  # // Added PR-AUC
            cv_scores["train_f1"].append(f1_score(y_tr, y_tr_pred, zero_division=0))
            cv_scores["train_recall"].append(recall_score(y_tr, y_tr_pred, zero_division=0))
            cv_scores["train_precision"].append(precision_score(y_tr, y_tr_pred, zero_division=0))

        cv_scores = {k: np.array(v) for k, v in cv_scores.items()}

    else:

        # Perform cross-validation
        cv_scores = cross_validate(
            model, X_for_cv, y_train_val,
            cv=cv_strategy,
            scoring=scoring_metrics,
            n_jobs=-1,
            return_train_score=True
        )

    # Store results
    cv_results[model_name] = cv_scores
    
    # Print elapsed time
    elapsed_time = time.time() - start_time
    print(f"Cross-validation for {model_name} completed in {elapsed_time:.2f} seconds!")


## 5. Model Comparison Dashboard

**Purpose**: Create comprehensive visualizations and tables comparing all models' cross-validation performance, including overfitting analysis and stability assessment.

### 5.1. Show simple table of metrics

In [ ]:
# Create performance comparison table
comparison_data = []

for model_name in models.keys():
    scores = cv_results[model_name]
    
    comparison_data.append({
        'Model': model_name,
        'PR AUC': f"{scores['test_pr_auc'].mean():.3f}",        # // Added PR-AUC as primary global metric
        'ROC AUC': f"{scores['test_roc_auc'].mean():.3f}",
        'Recall': f"{scores['test_recall'].mean():.3f}",
        'Precision': f"{scores['test_precision'].mean():.3f}",
        'F1 Score': f"{scores['test_f1'].mean():.3f}",
        'Accuracy': f"{scores['test_accuracy'].mean():.3f}",
        'Accuracy vs Baseline': f"{(scores['test_accuracy'].mean() - baseline_accuracy):.3f}",
    })

comparison_df = pd.DataFrame(comparison_data)
print("Model Performance Comparison:")
print("=" * 120)
print(comparison_df.to_string(index=False))
print("=" * 120)


### 5.2 Create dashboard

In [ ]:
# Create comprehensive cross-validation performance visualization
fig, axes = plt.subplots(4, 2, figsize=(16, 24))  # // Changed from (3, 2) to (4, 2) to add PR-AUC without replacing plots
fig.suptitle('Cross-Validation Performance: All Models Comparison', fontsize=16, fontweight='bold')

model_names_list = list(models.keys())

# Extract cross-validation metrics
cv_accuracies = [cv_results[name]['test_accuracy'].mean() for name in model_names_list]
cv_roc_aucs = [cv_results[name]['test_roc_auc'].mean() for name in model_names_list]
cv_recalls = [cv_results[name]['test_recall'].mean() for name in model_names_list]
cv_f1s = [cv_results[name]['test_f1'].mean() for name in model_names_list]
cv_precisions = [cv_results[name]['test_precision'].mean() for name in model_names_list]
cv_pr_aucs = [cv_results[name]['test_pr_auc'].mean() for name in model_names_list]  # // Added PR-AUC means

cv_accuracy_stds = [cv_results[name]['test_accuracy'].std() for name in model_names_list]
cv_roc_auc_stds = [cv_results[name]['test_roc_auc'].std() for name in model_names_list]
cv_recall_stds = [cv_results[name]['test_recall'].std() for name in model_names_list]
cv_f1_stds = [cv_results[name]['test_f1'].std() for name in model_names_list]
cv_precision_stds = [cv_results[name]['test_precision'].std() for name in model_names_list]
cv_pr_auc_stds = [cv_results[name]['test_pr_auc'].std() for name in model_names_list]  # // Added PR-AUC stds

# Extra robustness: convert NaNs (can occur if any fold metric becomes undefined) to 0.0 for plotting stability.
cv_accuracies = np.nan_to_num(cv_accuracies, nan=0.0)              
cv_roc_aucs = np.nan_to_num(cv_roc_aucs, nan=0.0)                  
cv_recalls = np.nan_to_num(cv_recalls, nan=0.0)                    
cv_f1s = np.nan_to_num(cv_f1s, nan=0.0)                            
cv_precisions = np.nan_to_num(cv_precisions, nan=0.0)              
cv_pr_aucs = np.nan_to_num(cv_pr_aucs, nan=0.0)                    # // Added PR-AUC NaN guard

cv_accuracy_stds = np.nan_to_num(cv_accuracy_stds, nan=0.0)        
cv_roc_auc_stds = np.nan_to_num(cv_roc_auc_stds, nan=0.0)          
cv_recall_stds = np.nan_to_num(cv_recall_stds, nan=0.0)            
cv_f1_stds = np.nan_to_num(cv_f1_stds, nan=0.0)                    
cv_precision_stds = np.nan_to_num(cv_precision_stds, nan=0.0)      
cv_pr_auc_stds = np.nan_to_num(cv_pr_auc_stds, nan=0.0)            # // Added PR-AUC NaN guard

# Plot 1: Recall (Row 1, Col 1)
bars1 = axes[0, 0].bar(model_names_list, cv_recalls, yerr=cv_recall_stds, capsize=5, alpha=0.8, color='lightgreen')
axes[0, 0].set_title('Recall', fontweight='bold')
axes[0, 0].set_ylabel('Recall')
axes[0, 0].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars1, cv_recalls, cv_recall_stds):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 2: F1 Score (Row 1, Col 2)
bars2 = axes[0, 1].bar(model_names_list, cv_f1s, yerr=cv_f1_stds, capsize=5, alpha=0.8, color='plum')
axes[0, 1].set_title('F1 Score', fontweight='bold')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars2, cv_f1s, cv_f1_stds):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 3: Precision (Row 2, Col 1)
bars3 = axes[1, 0].bar(model_names_list, cv_precisions, yerr=cv_precision_stds, capsize=5, alpha=0.8, color='salmon')
axes[1, 0].set_title('Precision', fontweight='bold')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars3, cv_precisions, cv_precision_stds):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 4: Accuracy (Row 2, Col 2)
bars4 = axes[1, 1].bar(model_names_list, cv_accuracies, yerr=cv_accuracy_stds, capsize=5, alpha=0.8, color='skyblue')
axes[1, 1].axhline(y=baseline_accuracy, color='red', linestyle='--', alpha=0.7, label=f'Baseline: {baseline_accuracy:.3f}')
axes[1, 1].set_title('Accuracy', fontweight='bold')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars4, cv_accuracies, cv_accuracy_stds):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.005, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 5: ROC AUC (Row 3, Col 1)
bars5 = axes[2, 0].bar(model_names_list, cv_roc_aucs, yerr=cv_roc_auc_stds, capsize=5, alpha=0.8, color='lightcoral')
axes[2, 0].axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Random: 0.500')
axes[2, 0].set_title('ROC AUC', fontweight='bold')
axes[2, 0].set_ylabel('ROC AUC')
axes[2, 0].legend()
axes[2, 0].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars5, cv_roc_aucs, cv_roc_auc_stds):
    axes[2, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 6: PR AUC (Row 3, Col 2)  # // Added directly after ROC-AUC
bars6 = axes[2, 1].bar(model_names_list, cv_pr_aucs, yerr=cv_pr_auc_stds, capsize=5,
                       alpha=0.8, color='mediumpurple')
axes[2, 1].axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Random: 0.500')  # // Added baseline
axes[2, 1].set_title('PR AUC (Average Precision)', fontweight='bold')
axes[2, 1].set_ylabel('PR AUC')
axes[2, 1].legend()
axes[2, 1].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars6, cv_pr_aucs, cv_pr_auc_stds):
    axes[2, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 7: Train vs Validation Accuracy (Row 4, Col 1)
cv_train_accuracies = [cv_results[name]['train_accuracy'].mean() for name in model_names_list]
x_pos = np.arange(len(model_names_list))
width = 0.35
bars7_train = axes[3, 0].bar(x_pos - width/2, cv_train_accuracies, width,
                             label='Training', alpha=0.8, color='steelblue')
bars7_val = axes[3, 0].bar(x_pos + width/2, cv_accuracies, width,
                           label='Validation', alpha=0.8, color='darkorange')
axes[3, 0].set_title('Training vs Validation Accuracy (Overfitting Check)', fontweight='bold')
axes[3, 0].set_ylabel('Accuracy')
axes[3, 0].set_xticks(x_pos)
axes[3, 0].set_xticklabels(model_names_list, rotation=45)
axes[3, 0].legend()
for i, (train_acc, val_acc) in enumerate(zip(cv_train_accuracies, cv_accuracies)):
    gap = train_acc - val_acc
    axes[3, 0].text(
        i, max(train_acc, val_acc) + 0.02, f'Gap: {gap:.3f}',
        ha='center', va='bottom', fontsize=9,
        color='red' if gap > OVERFITTING_THRESHOLD else 'green'
    )

axes[3, 1].set_visible(False)  # // Keep layout clean

plt.tight_layout()
plt.subplots_adjust(hspace=0.4)  # // Added: more vertical space so x-axis labels don't overlap
plt.show()  # // Added: explicitly render the figure

### 5.3 Perform statistical testing to compare scores

In [ ]:
# Statistical Significance Testing: Pairwise Model Comparison to determine 
# if differences in performance are statistically significant.
# Using Wilcoxon signed-rank test (non-parametric, appropriate for paired CV scores)

# Define significance level and apply Bonferroni correction for multiple comparisons
ALPHA = 0.05
model_names = list(models.keys())
n_comparisons = len(list(combinations(model_names, 2)))  # C(5,2) = 10 pairs
ALPHA_CORRECTED = ALPHA / n_comparisons  # Bonferroni correction

print("STATISTICAL SIGNIFICANCE ANALYSIS")
print("=" * 80)
print(f"Test: Wilcoxon signed-rank test (non-parametric, paired samples)")
print(f"Number of CV folds × repeats: {N_FOLDS} × {N_REPEATS} = {N_FOLDS * N_REPEATS} paired observations")
print(f"Number of pairwise comparisons: {n_comparisons}")
print(f"Significance level (α): {ALPHA}")
print(f"Bonferroni-corrected α: {ALPHA_CORRECTED:.4f}")
print()

# Metrics to test
metrics_to_test = ['test_recall']

# Store results for each metric
significance_results = {}

for metric in metrics_to_test:
    metric_name = metric.replace('test_', '').upper()
    print(f"\n{'─' * 80}")
    print(f"METRIC: {metric_name}")
    print(f"{'─' * 80}")
    
    # Create results storage
    pairwise_results = []
    
    # Perform pairwise comparisons
    for model_a, model_b in combinations(model_names, 2):
        scores_a = cv_results[model_a][metric]
        scores_b = cv_results[model_b][metric]
        
        # Calculate means
        mean_a = np.nanmean(scores_a)
        mean_b = np.nanmean(scores_b)
        diff = mean_a - mean_b
        
        # Perform Wilcoxon test
        try:
            stat, p_value = wilcoxon(scores_a, scores_b)
        except ValueError as e:
            # Can occur if all differences are zero
            stat, p_value = np.nan, 1.0
        
        # Determine significance
        sig_uncorrected = p_value < ALPHA
        sig_corrected = p_value < ALPHA_CORRECTED
        
        pairwise_results.append({
            'Model_A': model_a,
            'Model_B': model_b,
            'Mean_A': mean_a,
            'Mean_B': mean_b,
            'Difference': diff,
            'Statistic': stat,
            'p_value': p_value,
            'Significant_α0.05': sig_uncorrected,
            'Significant_Bonferroni': sig_corrected
        })
    
    # Create DataFrame and display
    results_df = pd.DataFrame(pairwise_results)
    significance_results[metric] = results_df
    
    # Display results
    print(f"\n{'Model A':<22} vs {'Model B':<22} | {'Diff':>7} | {'p-value':>10} | Sig (α=.05) | Sig (Bonf)")
    print("-" * 100)
    
    for _, row in results_df.iterrows():
        sig_mark = "✓" if row['Significant_α0.05'] else " "
        bonf_mark = "✓" if row['Significant_Bonferroni'] else " "
        winner = "←" if row['Difference'] > 0 else "→"
        
        print(f"{row['Model_A']:<22} vs {row['Model_B']:<22} | {row['Difference']:>+.4f} | {row['p_value']:>10.4f} |      {sig_mark}      |     {bonf_mark}")


## 6. Generalization of models
**Purpose**: Evaluate all models on the held-out test set to assess generalization performance.


In [ ]:
# TEST SET EVALUATION - FINAL MODEL PERFORMANCE ASSESSMENT

# Prepare test data
X_test = test_df[feature_cols]
y_test = test_df[target_col]

# Train final models on full train+val set for test evaluation
test_final_models = {}

for model_name, model in models.items():
    # Extract final training data
    X_final = X_train_val
    y_final = y_train_val

    if model_name == "XGBoost":
        # XGBoost was configured with fold-specific scale_pos_weight during CV.
        # For final training (train+val), compute scale_pos_weight once on y_final and inject it here.
        neg = (y_final == 0).sum()
        pos = (y_final == 1).sum()
        spw_final = (neg / pos) if pos > 0 else 1.0  # // Added zero-division guard

        trained_model = clone(model).set_params(scale_pos_weight=spw_final).fit(X_final, y_final)
        # Clone to ensure a fresh unfitted estimator and apply final (train+val) class imbalance weight.
    elif model_name == "Neural Network":
        #  In CV, MLP was trained with SMOTE on the training fold only.
        #  For final training, we apply SMOTE to the full train+val set.
        sm = SMOTE(random_state=42)
        X_res, y_res = sm.fit_resample(X_final, y_final)
        trained_model = clone(model).fit(X_res, y_res)
        #  Clone ensures a fresh model; SMOTE applied only on training data.
    else:
        trained_model = clone(model).fit(X_final, y_final)
        #  Clone is safer here too (fresh estimator), especially since we are reusing model objects across steps.

    test_final_models[model_name] = trained_model

# Dictionary to store test results
test_results = {}

# Evaluate each model on test set
for model_name, model in test_final_models.items():

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, zero_division=0)
    precision = precision_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Calculate ROC AUC + PR AUC if possible
    try:
        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_test)[:, 1]
        else:
            y_proba = model.decision_function(X_test)

        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)  # // Added PR-AUC (Average Precision)
    except Exception as e:
        roc_auc = None
        pr_auc = None  # // Added PR-AUC fallback

    # Store results
    test_results[model_name] = {
        'Accuracy': accuracy,
        'Recall': recall,
        'Precision': precision,
        'F1': f1,
        'ROC_AUC': roc_auc,
        'PR_AUC': pr_auc  # // Added PR-AUC to test results
    }

# Create test results summary table
print(f"\n" + "=" * 80)
print("TEST SET PERFORMANCE SUMMARY")
print("=" * 80)

test_df_results = pd.DataFrame(test_results).T
test_df_results = test_df_results.round(4)

print(test_df_results.to_string())

# Compare with cross-validation results
print(f"\n" + "=" * 80)
print("TEST vs CROSS-VALIDATION PERFORMANCE COMPARISON")
print("=" * 80)

comparison_results = []
for model_name in test_results.keys():
    if model_name in cv_results:
        cv_accuracy = cv_results[model_name]['test_accuracy'].mean()
        cv_recall = cv_results[model_name]['test_recall'].mean()
        cv_pr_auc = cv_results[model_name]['test_pr_auc'].mean()  # // Added PR-AUC CV mean

        test_accuracy = test_results[model_name]['Accuracy']
        test_recall = test_results[model_name]['Recall']
        test_pr_auc = test_results[model_name]['PR_AUC']  # // Added PR-AUC test

        accuracy_diff = test_accuracy - cv_accuracy
        recall_diff = test_recall - cv_recall
        pr_auc_diff = (test_pr_auc - cv_pr_auc) if (test_pr_auc is not None) else None  # // Added safe diff

        comparison_results.append({
            'Model': model_name,
            'CV_Accuracy': cv_accuracy,
            'Test_Accuracy': test_accuracy,
            'Accuracy_Diff': accuracy_diff,
            'CV_Recall': cv_recall,
            'Test_Recall': test_recall,
            'Recall_Diff': recall_diff,
            'CV_PR_AUC': cv_pr_auc,          # // Added
            'Test_PR_AUC': test_pr_auc,      # // Added
            'PR_AUC_Diff': pr_auc_diff       # // Added
        })

comparison_df = pd.DataFrame(comparison_results).round(4)

print("CV vs Test Performance (Diff = Test - CV):")
print(comparison_df.to_string(index=False))

# Performance analysis
print(f"\n" + "=" * 60)
print("TEST SET ANALYSIS & INSIGHTS")
print("=" * 60)

# Find best performing models on test set
best_accuracy_model = test_df_results['Accuracy'].idxmax()
best_recall_model = test_df_results['Recall'].idxmax()
best_f1_model = test_df_results['F1'].idxmax()
best_pr_auc_model = test_df_results['PR_AUC'].idxmax()  # // Added PR-AUC best model

print(f"\nBest Performance on Test Set:")
print(f"  🎯 Best Accuracy: {best_accuracy_model} ({test_df_results.loc[best_accuracy_model, 'Accuracy']:.4f})")
print(f"  🎯 Best Recall: {best_recall_model} ({test_df_results.loc[best_recall_model, 'Recall']:.4f})")
print(f"  🎯 Best F1 Score: {best_f1_model} ({test_df_results.loc[best_f1_model, 'F1']:.4f})")
print(f"  🎯 Best PR-AUC: {best_pr_auc_model} ({test_df_results.loc[best_pr_auc_model, 'PR_AUC']:.4f})")  # // Added

# Generalization assessment based on recall differences
print(f"\nGeneralization Assessment:")
for _, row in comparison_df.iterrows():
    model_name = row['Model']
    acc_diff = row['Accuracy_Diff']
    recall_diff = row['Recall_Diff']

    if abs(recall_diff) < 0.02:
        generalization = "✅ Excellent"
    elif abs(recall_diff) < 0.05:
        generalization = "🟡 Good"
    else:
        generalization = "⚠️  Concerning"

    print(f"  • {model_name}: {generalization} (Recall: {recall_diff:+.3f}, Accuracy: {acc_diff:+.3f})")


## 7. Feature Importance Comparison

**Purpose**: Compare feature importance/coefficients across all models to understand which features are consistently important.

In [ ]:
# Extract feature importance/coefficients from each model
feature_importance = pd.DataFrame({
    'Feature': feature_cols
})

for model_name, model in test_final_models.items():
    if model_name == 'Logistic Regression':
        # Get coefficients (absolute values for importance)
        importance = np.abs(model.coef_[0])
        feature_importance[f'{model_name}_Importance'] = importance
        
    elif model_name in ['Decision Tree', 'Random Forest', 'XGBoost']:
        # Get feature importance
        importance = model.feature_importances_
        feature_importance[f'{model_name}_Importance'] = importance
        
    elif model_name == 'Neural Network':
        # For neural networks, we'll use a simple approximation
        # This is less interpretable but gives some indication
        if hasattr(model, 'coefs_'):
            # Sum of absolute weights from input to first hidden layer
            importance = np.abs(model.coefs_[0]).sum(axis=1)
            feature_importance[f'{model_name}_Importance'] = importance

# Normalize importance scores to 0-1 scale for comparison
importance_cols = [col for col in feature_importance.columns if 'Importance' in col]
for col in importance_cols:
    if col in feature_importance.columns:
        max_val = feature_importance[col].max()
        if max_val > 0:
            feature_importance[col] = feature_importance[col] / max_val

# Calculate average importance across models
feature_importance['Average_Importance'] = feature_importance[importance_cols].mean(axis=1)

# Sort by average importance
feature_importance = feature_importance.sort_values('Average_Importance', ascending=False)

In [ ]:
# Extract feature importance/coefficients from each model
feature_importance = pd.DataFrame({
    'Feature': feature_cols
})

for model_name, model in test_final_models.items():
    if model_name == 'Logistic Regression':
        # Get coefficients (absolute values for importance)
        importance = np.abs(model.coef_[0])
        feature_importance[f'{model_name}_Importance'] = importance

    elif model_name in ['Decision Tree', 'Random Forest', 'XGBoost']:
        # Get feature importance
        importance = model.feature_importances_
        feature_importance[f'{model_name}_Importance'] = importance

    elif model_name == 'Neural Network':
        # For neural networks, we'll use a simple approximation
        # This is less interpretable but gives some indication
        if hasattr(model, 'coefs_'):
            # Sum of absolute weights from input to first hidden layer
            importance = np.abs(model.coefs_[0]).sum(axis=1)
            feature_importance[f'{model_name}_Importance'] = importance

# Ensure all importance columns exist before using them
importance_cols = [col for col in feature_importance.columns if col.endswith('_Importance')]  

# Normalize importance scores to 0-1 scale for comparison
for col in importance_cols:
    max_val = feature_importance[col].max()
    if pd.notna(max_val) and max_val > 0:  
        feature_importance[col] = feature_importance[col] / max_val

# Compute average only across columns that actually exist (avoid empty list / all-NaN surprises).
if len(importance_cols) > 0:  
    feature_importance['Average_Importance'] = feature_importance[importance_cols].mean(axis=1)
else:
    feature_importance['Average_Importance'] = np.nan  

# Sort by average importance
feature_importance = feature_importance.sort_values('Average_Importance', ascending=False)

In [ ]:
# Visualize feature importance comparison
fig, axes = plt.subplots(2, 3, figsize=(24, 16))
fig.suptitle('Feature Importance Comparison Across Models', fontsize=16, fontweight='bold')

# Get top 10 features for visualization
top_10_features = feature_importance.head(10)

# Plot each model's feature importance
model_cols = [c for c in feature_importance.columns if c.endswith('_Importance') and c != 'Average_Importance']  

# Ensure we only plot columns that exist and have at least some non-null values 
model_cols = [c for c in model_cols if c in feature_importance.columns and feature_importance[c].notna().any()]  

max_plots = min(5, len(model_cols))  # Limit to 5 models, but respect actual available columns.

for i in range(max_plots):
    imp_col = model_cols[i] 

    row = i // 3
    col_idx = i % 3

    # Create horizontal bar plot
    y_pos = np.arange(len(top_10_features))
    vals = top_10_features[imp_col].fillna(0.0)  # Fill NaNs so plotting doesn't break.

    axes[row, col_idx].barh(y_pos, vals, alpha=0.7)
    axes[row, col_idx].set_yticks(y_pos)
    axes[row, col_idx].set_yticklabels([name[:30] for name in top_10_features['Feature']], fontsize=10)
    axes[row, col_idx].set_xlabel('Normalized Importance', fontsize=11)
    axes[row, col_idx].set_title(imp_col.replace('_Importance', ''), fontsize=12, fontweight='bold')
    axes[row, col_idx].grid(axis='x', alpha=0.3)

    # Invert y-axis to show most important at top
    axes[row, col_idx].invert_yaxis()

# Hide any unused subplots automatically (not just the [1,2] case).
for j in range(max_plots, 6):  # 2x3 grid = 6 axes total
    r = j // 3
    c = j % 3
    axes[r, c].set_visible(False)

plt.tight_layout()
plt.show()


## 8. Best Model Selection & Comprehensive Analysis

**Purpose**: Select the best performing model based on:
- PR-AUC: primary selector
- Secondary tie-breaker / refinement: PR-AUC + ROC-AUC

In [ ]:
# ======================================================================
# PURPOSE: MODEL SELECTION BASED ON PR-AUC WITH ROC-AUC AS SECONDARY CRITERION
# ----------------------------------------------------------------------
# This block performs cross-validated model ranking and selection with a
# focus on intervention effectiveness for dropout prediction.
#
# PRIMARY METRIC — PR-AUC (Average Precision):
# • PR-AUC is used as the primary model-selection metric.
# • It evaluates the precision–recall trade-off across *all thresholds*,
#   explicitly accounting for class imbalance and false positives.
# • It directly answers the operational question:
#   "When the model flags a student as at risk, how often is that correct?"
# • This makes PR-AUC the most appropriate global metric for intervention-
#   driven use cases where resources are limited.
#
# SECONDARY METRIC — ROC-AUC:
# • ROC-AUC is used as a secondary criterion to assess overall ranking
#   quality (i.e., how well dropouts are scored above non-dropouts).
# • While threshold-independent, ROC-AUC does not consider precision and
#   can remain high even when false positives are excessive.
# • It is therefore used as a stability and discrimination sanity check,
#   not as the primary decision metric.
#
# WHY OTHER METRICS ARE NOT USED FOR MODEL SELECTION:
# • Accuracy is threshold-dependent and can be misleading in the presence
#   of class imbalance, even when imbalance mitigation is applied.
# • Recall alone ignores precision and may favor models that over-flag
#   students, which is impractical for real interventions.
# • Precision alone ignores recall and may miss many at-risk students.
# • F1 score is threshold-dependent and reflects performance at a single
#   operating point rather than across all decision thresholds.
#
# SELECTION STRATEGY:
# 1) Rank models by mean cross-validated PR-AUC
# 2) Use ROC-AUC jointly with PR-AUC to break ties and assess robustness
# 3) Report Accuracy, Recall, and F1 for interpretability only
#
# This approach prioritizes models that support reliable, actionable
# dropout interventions rather than purely global discrimination.
# ======================================================================


In [ ]:
# COMPREHENSIVE MODEL COMPARISON AND BEST MODEL SELECTION
print("COMPREHENSIVE MODEL COMPARISON ANALYSIS")
print("=" * 80)
print(f"Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Dataset: {df.shape[0]:,} students, {len(feature_cols)} features")
print(f"Evaluation Method: Stratified {N_FOLDS}-fold cross-validation")

# 1. MODEL RANKINGS AND SELECTION
print("\n1. MODEL RANKINGS BASED ON CROSS-VALIDATION:")
print("-" * 50)

model_rankings_cv = []
for model_name in models.keys():
    scores = cv_results[model_name]

    # Nan-safe aggregation
    pr_auc_mean = np.nanmean(scores['test_pr_auc'])      # Primary metric
    pr_auc_std  = np.nanstd(scores['test_pr_auc'])

    roc_auc_mean = np.nanmean(scores['test_roc_auc'])    # Secondary metric
    roc_auc_std  = np.nanstd(scores['test_roc_auc'])

    acc_mean = np.nanmean(scores['test_accuracy'])
    acc_std  = np.nanstd(scores['test_accuracy'])

    recall_mean = np.nanmean(scores['test_recall'])
    recall_std  = np.nanstd(scores['test_recall'])

    f1_mean = np.nanmean(scores['test_f1'])
    f1_std  = np.nanstd(scores['test_f1'])

    model_rankings_cv.append({
        'Model': model_name,
        'CV_PR_AUC_Mean': pr_auc_mean,
        'CV_PR_AUC_Std': pr_auc_std,
        'CV_ROC_AUC_Mean': roc_auc_mean,
        'CV_ROC_AUC_Std': roc_auc_std,
        'CV_Accuracy_Mean': acc_mean,
        'CV_Accuracy_Std': acc_std,
        'CV_Recall_Mean': recall_mean,
        'CV_Recall_Std': recall_std,
        'CV_F1_Mean': f1_mean,
        'CV_F1_Std': f1_std,
        # Secondary combined signal (only used after PR-AUC)
        'PR_ROC_Combined': (pr_auc_mean + roc_auc_mean) / 2,
        # Optional: keep baseline improvement like your old code
        'Improvement_Over_Baseline': acc_mean - baseline_accuracy
    })

cv_ranking_df = pd.DataFrame(model_rankings_cv)

# Ranking rule:
#   1) Highest PR-AUC
#   2) Highest (PR-AUC + ROC-AUC) / 2 as tie-breaker
cv_ranking_df = cv_ranking_df.sort_values(
    ['CV_PR_AUC_Mean', 'PR_ROC_Combined'],
    ascending=[False, False]
).reset_index(drop=True)

print("Rankings based on PR-AUC (primary) with ROC-AUC as secondary signal:")

# Display rankings
for rank, row in enumerate(cv_ranking_df.itertuples(index=False), start=1):
    emoji = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "📊"

    print(f"{emoji} {rank}. {row.Model}:")
    print(f"     PR-AUC: {row.CV_PR_AUC_Mean:.3f} (±{row.CV_PR_AUC_Std:.3f})")
    print(f"     ROC-AUC: {row.CV_ROC_AUC_Mean:.3f} (±{row.CV_ROC_AUC_Std:.3f})")
    print(f"     PR+ROC Combined: {row.PR_ROC_Combined:.3f}")
    print(f"     Recall: {row.CV_Recall_Mean:.3f} (±{row.CV_Recall_Std:.3f}) | "
          f"F1: {row.CV_F1_Mean:.3f} (±{row.CV_F1_Std:.3f})")
    print(f"     Accuracy: {row.CV_Accuracy_Mean:.3f} (±{row.CV_Accuracy_Std:.3f})")
    print(f"     Improvement over baseline: +{row.Improvement_Over_Baseline:.3f}")

# ✅ Select best model (THIS is what your current code is missing)
best_model_name = cv_ranking_df.iloc[0]['Model']

print(f"\n🏆 BEST MODEL SELECTED: {best_model_name}")
print(f"   • CV PR-AUC: {cv_ranking_df.iloc[0]['CV_PR_AUC_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_PR_AUC_Std']:.3f})")
print(f"   • CV ROC-AUC: {cv_ranking_df.iloc[0]['CV_ROC_AUC_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_ROC_AUC_Std']:.3f})")
print(f"   • PR+ROC Combined: {cv_ranking_df.iloc[0]['PR_ROC_Combined']:.3f}")
print(f"   • CV Accuracy: {cv_ranking_df.iloc[0]['CV_Accuracy_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_Accuracy_Std']:.3f})")
print(f"   • CV Recall: {cv_ranking_df.iloc[0]['CV_Recall_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_Recall_Std']:.3f})")
print(f"   • CV F1 Score: {cv_ranking_df.iloc[0]['CV_F1_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_F1_Std']:.3f})")
print(f"   • Improvement over baseline: +{cv_ranking_df.iloc[0]['Improvement_Over_Baseline']:.3f}")

# Store the ranking dataframe for later use (updated columns for your PR-AUC ranking)
ranking_df = cv_ranking_df.copy()
ranking_df.columns = [
    'Model',
    'PR_AUC_Mean', 'PR_AUC_Std',
    'ROC_AUC_Mean', 'ROC_AUC_Std',
    'Accuracy_Mean', 'Accuracy_Std',
    'Recall_Mean', 'Recall_Std',
    'F1_Mean', 'F1_Std',
    'PR_ROC_Combined',
    'Improvement_Over_Baseline'
]


XGBoost was selected as the final model as it achieved the highest cross-validated PR-AUC, 

indicating the best precision–recall trade-off for dropout identification, 

while also demonstrating strong overall ranking performance via ROC-AUC. 

It is notable that Logistic Regression performs very close to XGBoost across PR-AUC, ROC-AUC, and recall. 

This is a positive indication of model robustness rather than a weakness: it suggests that the underlying signal is strong, 

the selected features are informative, and the problem structure is largely learnable with simpler models. 

XGBoost’s consistent but modest improvement over Logistic Regression indicates that it is capturing additional non-linear interactions without overfitting,

reinforcing confidence in both the feature engineering and the final model choice.


In [ ]:
# 2. DETAILED ANALYSIS OF BEST MODEL
print(f"\n2. DETAILED ANALYSIS OF BEST MODEL: {best_model_name}")
print("-" * 60)

# Get cross-validation performance for comprehensive analysis
best_cv_scores = cv_results[best_model_name]

# Use nan-safe summaries to avoid edge cases where a fold metric becomes undefined (rare, but possible).
val_acc_mean = np.nanmean(best_cv_scores['test_accuracy'])          
val_acc_std  = np.nanstd(best_cv_scores['test_accuracy'])           
tr_acc_mean  = np.nanmean(best_cv_scores['train_accuracy'])         
tr_acc_std   = np.nanstd(best_cv_scores['train_accuracy'])          
val_auc_mean = np.nanmean(best_cv_scores['test_roc_auc'])           
val_auc_std  = np.nanstd(best_cv_scores['test_roc_auc'])            
val_rec_mean = np.nanmean(best_cv_scores['test_recall'])            
val_rec_std  = np.nanstd(best_cv_scores['test_recall'])             
val_f1_mean  = np.nanmean(best_cv_scores['test_f1'])                
val_f1_std   = np.nanstd(best_cv_scores['test_f1'])                 

print("\nPerformance Metrics:")
print(f"  • Validation Accuracy: {val_acc_mean:.4f} (±{val_acc_std:.4f})")  
print(f"  • Training Accuracy: {tr_acc_mean:.4f} (±{tr_acc_std:.4f})")      
print(f"  • Validation ROC AUC: {val_auc_mean:.4f} (±{val_auc_std:.4f})")   
print(f"  • Validation Recall: {val_rec_mean:.4f} (±{val_rec_std:.4f})")    
print(f"  • Validation F1 Score: {val_f1_mean:.4f} (±{val_f1_std:.4f})")    

# Overfitting and stability analysis
train_accuracy = tr_acc_mean                                           
cv_accuracy = val_acc_mean                                              
cv_roc_auc = val_auc_mean                                               
cv_recall = val_rec_mean                                                
cv_f1 = val_f1_mean                                                     
accuracy_gap = train_accuracy - cv_accuracy
acc_std = val_acc_std                                                   
auc_std = val_auc_std                                                   

print(f"\nModel Quality Assessment:")
print(f"  • Training-Validation gap: {accuracy_gap:+.4f}")

# Assess overfitting
if accuracy_gap < STABILITY_THRESHOLD:
    overfitting = "✅ No overfitting detected"
elif accuracy_gap < OVERFITTING_THRESHOLD:
    overfitting = "🟡 Slight overfitting"
else:
    overfitting = "⚠️  Potential overfitting"

# Assess stability
if acc_std < STABILITY_THRESHOLD and auc_std < STABILITY_THRESHOLD:
    stability = "✅ Very stable"
elif acc_std < (2 * STABILITY_THRESHOLD) and auc_std < (2 * STABILITY_THRESHOLD): 
    stability = "🟡 Moderately stable"
else:
    stability = "⚠️  Less stable"

# Performance rating
if cv_roc_auc >= 0.8:
    auc_rating = "Excellent"
elif cv_roc_auc >= 0.7:
    auc_rating = "Good"
elif cv_roc_auc >= 0.6:
    auc_rating = "Fair"
else:
    auc_rating = "Poor"

# Recall assessment for class imbalance
if cv_recall >= 0.8:
    recall_rating = "Excellent"
elif cv_recall >= 0.7:
    recall_rating = "Good"
elif cv_recall >= 0.6:
    recall_rating = "Fair"
else:
    recall_rating = "Poor"

print(f"  • Overfitting: {overfitting}")
print(f"  • Stability: {stability}")
print(f"  • Performance Rating: {auc_rating} (ROC AUC: {cv_roc_auc:.3f})")
print(f"  • Recall Rating: {recall_rating} (Recall: {cv_recall:.3f}) - Important for identifying at-risk students")
print(f"  • Improvement over baseline: {(cv_accuracy - baseline_accuracy)*100:.1f} percentage points")

# Model configuration details
print(f"\nModel Configuration:")
if best_model_name in models:
    best_model = models[best_model_name]
    print(f"  • Type: {type(best_model).__name__}")
    key_params = ['random_state', 'class_weight', 'n_estimators', 'max_depth', 'max_iter', 'early_stopping']
    params = best_model.get_params()

    relevant_params = {k: v for k, v in params.items() if k in key_params}

    if relevant_params:
        print(f"  • Key Parameters: {', '.join([f'{k}={v}' for k, v in relevant_params.items()])}")

# Store variables for backward compatibility
best_scores = best_cv_scores


In [ ]:
# 2. DETAILED ANALYSIS OF BEST MODEL
print(f"\n2. DETAILED ANALYSIS OF BEST MODEL: {best_model_name}")
print("-" * 60)

# Get cross-validation performance for comprehensive analysis
best_cv_scores = cv_results[best_model_name]

# Use nan-safe summaries to avoid edge cases where a fold metric becomes undefined (rare, but possible).
val_acc_mean = np.nanmean(best_cv_scores['test_accuracy'])
val_acc_std  = np.nanstd(best_cv_scores['test_accuracy'])
tr_acc_mean  = np.nanmean(best_cv_scores['train_accuracy'])
tr_acc_std   = np.nanstd(best_cv_scores['train_accuracy'])
val_auc_mean = np.nanmean(best_cv_scores['test_roc_auc'])
val_auc_std  = np.nanstd(best_cv_scores['test_roc_auc'])
val_pr_mean  = np.nanmean(best_cv_scores['test_pr_auc'])            # // Added PR-AUC (primary)
val_pr_std   = np.nanstd(best_cv_scores['test_pr_auc'])             # // Added PR-AUC (primary)
val_rec_mean = np.nanmean(best_cv_scores['test_recall'])
val_rec_std  = np.nanstd(best_cv_scores['test_recall'])
val_f1_mean  = np.nanmean(best_cv_scores['test_f1'])
val_f1_std   = np.nanstd(best_cv_scores['test_f1'])

print("\nPerformance Metrics:")
print(f"  • Validation PR-AUC: {val_pr_mean:.4f} (±{val_pr_std:.4f})")      # // Added PR-AUC first (primary)
print(f"  • Validation ROC AUC: {val_auc_mean:.4f} (±{val_auc_std:.4f})")
print(f"  • Validation Accuracy: {val_acc_mean:.4f} (±{val_acc_std:.4f})")
print(f"  • Validation Recall: {val_rec_mean:.4f} (±{val_rec_std:.4f})")
print(f"  • Validation F1 Score: {val_f1_mean:.4f} (±{val_f1_std:.4f})")
print(f"  • Training Accuracy: {tr_acc_mean:.4f} (±{tr_acc_std:.4f})")      # // Moved down to keep validation metrics grouped

# Overfitting and stability analysis
train_accuracy = tr_acc_mean
cv_accuracy = val_acc_mean
cv_roc_auc = val_auc_mean
cv_pr_auc = val_pr_mean                                                # // Added PR-AUC variable
cv_recall = val_rec_mean
cv_f1 = val_f1_mean
accuracy_gap = train_accuracy - cv_accuracy
acc_std = val_acc_std
auc_std = val_auc_std
pr_std = val_pr_std                                                    # // Added PR-AUC stability reference

# Secondary combined indicator (matches selection tie-break logic)
pr_roc_combined = (cv_pr_auc + cv_roc_auc) / 2                          # // Added combined signal

print(f"\nModel Quality Assessment:")
print(f"  • Training-Validation gap: {accuracy_gap:+.4f}")

# Assess overfitting
if accuracy_gap < STABILITY_THRESHOLD:
    overfitting = "✅ No overfitting detected"
elif accuracy_gap < OVERFITTING_THRESHOLD:
    overfitting = "🟡 Slight overfitting"
else:
    overfitting = "⚠️  Potential overfitting"

# Assess stability (use PR-AUC + ROC-AUC as primary stability signals, consistent with selection)
if pr_std < STABILITY_THRESHOLD and auc_std < STABILITY_THRESHOLD:      # // Changed: stability based on PR-AUC + ROC-AUC
    stability = "✅ Very stable"
elif pr_std < (2 * STABILITY_THRESHOLD) and auc_std < (2 * STABILITY_THRESHOLD):  # // Changed
    stability = "🟡 Moderately stable"
else:
    stability = "⚠️  Less stable"

# Performance rating (primary = PR-AUC, secondary = ROC-AUC)
if cv_pr_auc >= 0.75:                                                   # // Changed: rating uses PR-AUC (primary)
    pr_rating = "Excellent"
elif cv_pr_auc >= 0.65:
    pr_rating = "Good"
elif cv_pr_auc >= 0.55:
    pr_rating = "Fair"
else:
    pr_rating = "Poor"

if cv_roc_auc >= 0.8:
    auc_rating = "Excellent"
elif cv_roc_auc >= 0.7:
    auc_rating = "Good"
elif cv_roc_auc >= 0.6:
    auc_rating = "Fair"
else:
    auc_rating = "Poor"

# Recall assessment
if cv_recall >= 0.8:
    recall_rating = "Excellent"
elif cv_recall >= 0.7:
    recall_rating = "Good"
elif cv_recall >= 0.6:
    recall_rating = "Fair"
else:
    recall_rating = "Poor"

print(f"  • Overfitting: {overfitting}")
print(f"  • Stability: {stability}")
print(f"  • Performance Rating: {pr_rating} (PR-AUC: {cv_pr_auc:.3f})")     # // Changed: PR-AUC is primary rating
print(f"  • Secondary Ranking Quality: {auc_rating} (ROC-AUC: {cv_roc_auc:.3f})")  # // Added ROC-AUC as secondary
print(f"  • PR+ROC Combined: {pr_roc_combined:.3f}")                        # // Added combined value
print(f"  • Recall Rating: {recall_rating} (Recall: {cv_recall:.3f}) - Important for identifying at-risk students")
print(f"  • Improvement over baseline: {(cv_accuracy - baseline_accuracy)*100:.1f} percentage points")

# Model configuration details
print(f"\nModel Configuration:")
if best_model_name in models:
    best_model = models[best_model_name]
    print(f"  • Type: {type(best_model).__name__}")
    key_params = ['random_state', 'class_weight', 'n_estimators', 'max_depth', 'max_iter', 'early_stopping']
    params = best_model.get_params()

    relevant_params = {k: v for k, v in params.items() if k in key_params}

    if relevant_params:
        print(f"  • Key Parameters: {', '.join([f'{k}={v}' for k, v in relevant_params.items()])}")

# Store variables for backward compatibility
best_scores = best_cv_scores


In [ ]:
# 3. KEY INSIGHTS AND PATTERNS
print(f"\n3. KEY INSIGHTS AND PATTERNS:")
print("-" * 40)

# Insight 1: Best performing model type
if best_model_name == 'Random Forest':
    insight1 = "Ensemble methods (Random Forest) perform best, indicating complex feature interactions."
elif best_model_name == 'XGBoost':
    insight1 = "Boosted trees (XGBoost) perform best, suggesting strong non-linear structure and additive interactions."  # Added explicit XGBoost case.
elif best_model_name == 'Neural Network':
    insight1 = "Neural networks capture non-linear patterns effectively for this problem."
elif best_model_name == 'Logistic Regression':
    insight1 = "Linear relationships are sufficient; simpler models work well."
else:
    insight1 = "Tree-based models provide good interpretability with solid performance."

print(f"• Model Type: {insight1}")

# Insight 2: Performance level and readiness
# // Use PR-AUC as the primary readiness signal (consistent with selection criteria), with ROC-AUC as supporting context.
if cv_pr_auc >= 0.70:  # // Changed from ROC-AUC to PR-AUC thresholds
    performance_level = "excellent"
    actionable = "Model is ready for further validation and potential deployment."
elif cv_pr_auc >= 0.60:  # // Changed
    performance_level = "good"
    actionable = "Model shows promise but could benefit from threshold tuning and calibration."
else:  # // Changed
    performance_level = "moderate"
    actionable = "Consider feature engineering, alternative models, or improved imbalance handling."

print(f"• Performance Level: Model achieves {performance_level} cross-validation performance (PR-AUC-focused).")  # // Changed wording
print(f"• Readiness: {actionable}")
print(f"  (Context: ROC-AUC = {cv_roc_auc:.3f})")  # // Added: keep ROC-AUC visible as secondary metric

# Insight 3: Feature importance consistency
if len(importance_cols) > 1:
    # Importance measures differ by model type (LR coefficients vs tree impurity vs NN weight proxy),
    # so this correlation is a heuristic indicator only (not a rigorous agreement measure).
    corr_matrix = feature_importance[importance_cols].corr()  # Store once for readability.
    upper = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)]

    avg_importance_corr = np.nanmean(upper)

    if avg_importance_corr > 0.7:
        consistency_insight = "High agreement between models on feature importance suggests robust feature signals."
    elif avg_importance_corr > 0.4:
        consistency_insight = "Moderate agreement between models on feature importance."
    else:
        consistency_insight = "Low agreement suggests different models emphasize different aspects of the data."

    print(f"• Feature Consistency: {consistency_insight}")
    print(f"  (Heuristic avg correlation across importance columns: {avg_importance_corr:.2f})")

print(f"\n4. TOP PREDICTIVE FEATURES:")
print("-" * 40)
top_20_features = feature_importance.head(20)
for i, (_, row) in enumerate(top_20_features.iterrows(), 1):
    feature_name = row['Feature'][:50]  # Truncate long names
    avg_importance = row['Average_Importance']
    print(f"{i}. {feature_name}: {avg_importance:.3f}")

print(f"\n5. RECOMMENDATIONS AND NEXT STEPS:")
print("-" * 40)

print(f"📋 Immediate Actions:")
if cv_pr_auc >= 0.60:  # // Changed from ROC-AUC gate to PR-AUC gate
    print(f"   ✅ Proceed with hyperparameter tuning for {best_model_name}")
    print(f"   ✅ Consider final validation on hold-out test set")
    print(f"   ✅ Prepare for potential deployment pipeline")
else:
    print(f"   ⚠️  Focus on feature engineering before further optimization")
    print(f"   ⚠️  Collect additional features or domain expertise")
    print(f"   ⚠️  Consider alternative modeling approaches")

print(f"\n📊 Model Development Priorities:")
print(f"   • Hyperparameter tuning focus: {best_model_name}")
print(f"   • Use cross-validation for parameter optimization")
print(f"   • Consider ensemble methods if multiple models perform similarly")

print(f"\n🔬 Long-term Improvements:")
print(f"   • Feature engineering based on top predictive features")
print(f"   • Advanced hyperparameter optimization (Bayesian, Grid Search)")
print(f"   • Ensemble of top-performing models")
print(f"   • Additional data collection for underperforming classes")

## 9. Save models and cv results

In [ ]:
# Define data to be saved
analysis_data = {
    'cv_results': cv_results,
    'models': models,
    'best_model_name': best_model_name,
    'feature_importance': feature_importance,
    'ranking_df': ranking_df,
    'baseline_accuracy': baseline_accuracy
}
# Save data
joblib.dump(analysis_data, PROCESSED_DIR / 'model_comparison_analysis.pkl')
print("\nModel comparison analysis data saved successfully!")